# C1-ml-fundamentals — Practice p03 — Solution

The cell below generates 60 labeled beetle body lengths (species `0` small,
species `1` large) and splits them 40 train / 20 test. Build the two
classifiers from the lesson and compare them.

1. **Memorizer.** Store every training length with its label in a lookup
   table. To predict: return the stored label on an exact match, otherwise
   return the most common training label. Compute its training accuracy as
   `mem_train_acc` and test accuracy as `mem_test_acc` (exact names).
2. **Single threshold.** Predict species 1 whenever the length is at least a
   cutoff `t`. Try every midpoint between neighbouring sorted training
   lengths, keep the cutoff with the best training accuracy, and compute
   `thr_train_acc` and `thr_test_acc`.
3. **Short answer** (markdown cell): which classifier is the more flexible
   one, what happened to it between train and test, and what is that
   phenomenon called?

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
SEED = 20260804
data_rng = np.random.default_rng(SEED)
len_small = data_rng.normal(4.0, 0.8, size=30)   # species 0
len_large = data_rng.normal(6.5, 0.8, size=30)   # species 1

X = np.concatenate([len_small, len_large])
y = np.concatenate([np.zeros(30, dtype=int), np.ones(30, dtype=int)])

order = data_rng.permutation(60)
X, y = X[order], y[order]
X_train, y_train = X[:40], y[:40]
X_test,  y_test  = X[40:], y[40:]

In [ ]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def memorizer_predict(X_tr, y_tr, X_new):
    majority = int(np.argmax(np.bincount(y_tr)))
    table = {float(v): int(lab) for v, lab in zip(X_tr, y_tr)}
    return np.array([table.get(float(v), majority) for v in X_new])

mem_train_acc = accuracy(y_train, memorizer_predict(X_train, y_train, X_train))
mem_test_acc  = accuracy(y_test,  memorizer_predict(X_train, y_train, X_test))
print("memorizer: train", mem_train_acc, " test", mem_test_acc)

In [ ]:
def choose_threshold(X_tr, y_tr):
    s = np.sort(X_tr)
    candidates = (s[:-1] + s[1:]) / 2
    best_t, best_acc = candidates[0], -1.0
    for t in candidates:
        acc = np.mean((X_tr >= t).astype(int) == y_tr)
        if acc > best_acc:
            best_t, best_acc = float(t), float(acc)
    return best_t

t = choose_threshold(X_train, y_train)
thr_train_acc = accuracy(y_train, (X_train >= t).astype(int))
thr_test_acc  = accuracy(y_test,  (X_test  >= t).astype(int))
print(f"threshold: cutoff {t:.3f}  train {thr_train_acc}  test {thr_test_acc}")

**Task 3 answer.** The memorizer is the more flexible classifier: it bends to
every single training example, which is why its training accuracy is a perfect
100%. On test data it collapses to 50%, because
none of the new lengths match its table and it can only fall back on the
majority guess. That large train-to-test gap is **overfitting** — the flexible
rule learned the examples, not the pattern, while the rigid-but-sensible
threshold keeps 95% on unseen beetles.

### Answer check

In [ ]:
assert np.isclose(mem_train_acc, 1.0, atol=1e-9, rtol=0)
assert np.isclose(mem_test_acc, 0.5, atol=1e-9, rtol=0)
assert np.isclose(thr_train_acc, 0.95, atol=1e-9, rtol=0)
assert np.isclose(thr_test_acc, 0.95, atol=1e-9, rtol=0)
assert mem_test_acc < thr_test_acc          # the memorizer overfits
assert mem_train_acc > thr_train_acc        # ...despite winning on training
print("p03 OK")